# School Email Merger

Merges school names with found emails using fuzzy matching.

In [ ]:
import pandas as pd
import re
from google.colab import files
from rapidfuzz import fuzz

print("Upload your school list file:")
up1 = files.upload()
df1 = pd.read_csv(list(up1.keys())[0])

print("\nUpload your scraped data file:")
up2 = files.upload()
df2 = pd.read_csv(list(up2.keys())[0])

STOP_WORDS = {'the', 'of', 'and', 'limited', 'co', 'ltd', 'inc', 'service', 'resource'}

def clean_name(text):
    if pd.isna(text):
        return ""
    text = re.sub(r'[^\w\s]', ' ', str(text).lower())
    return " ".join([w for w in text.split() if w not in STOP_WORDS])

def get_detailed_matches(target_name, lookup_df, threshold=85):
    cleaned_target = clean_name(target_name)
    if not cleaned_target:
        return "N/A"

    matched_details = []
    for _, row in lookup_df.iterrows():
        cleaned_org = clean_name(row['Organization'])
        score = fuzz.token_set_ratio(cleaned_target, cleaned_org)

        if score >= threshold:
            org_name = str(row['Organization']).strip()
            email = str(row['Email Address']).strip()
            matched_details.append(f"{org_name} ({email})")

    if matched_details:
        return " | ".join(list(set(matched_details)))
    return "No Match Found"

print("\nMatching and linking sources...")
df1['Match_Details'] = df1['NAME_EN'].apply(lambda x: get_detailed_matches(x, df2))

output_name = 'Verified_Matches_List.xlsx'
df1.to_excel(output_name, index=False)
print(f"\n Success! Downloading {output_name}")
files.download(output_name)